# 01 - Out of Sample validation 
## <b> Adrian Vazquez </b>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from scipy.stats import linregress
early_regime = pd.read_csv("../data/processed/early_regime_dataset.csv")
metadata = pd.read_csv("../data/raw/markets_metadata.csv")
metadata = metadata.rename(columns={"id": "market_id"})
df_alpha = pd.read_csv('../data/processed/regime_alpha_dataset.csv')
df_alpha = df_alpha.merge(metadata[["market_id", "endDate"]],on="market_id",how="left")

In [ ]:
df_alpha["endDate"] = pd.to_datetime(df_alpha["endDate"])
df_alpha = df_alpha.sort_values("endDate")
df_alpha[["market_id","endDate"]].head()

,market_id,endDate
1,249778,2023-04-02 00:00:00+00:00
2,250474,2023-04-26 00:00:00+00:00
0,248594,2023-05-01 00:00:00+00:00
4,251027,2023-05-22 00:00:00+00:00
6,251046,2023-05-28 00:00:00+00:00


In [ ]:
df_alpha[["market_id","endDate"]].tail()

,market_id,endDate
30,254573,2024-12-30 12:00:00+00:00
29,254572,2024-12-30 12:00:00+00:00
34,254579,2024-12-30 12:00:00+00:00
33,254578,2024-12-30 12:00:00+00:00
28,254571,2024-12-31 00:00:00+00:00


In [11]:
df_25 = early_regime[early_regime["horizon"] == 0.25]
df_25 = df_25.merge( df_alpha[["market_id", "endDate"]],on="market_id",how="left")

In [12]:
df_25.shape

(43, 34)

In [13]:
df_25["endDate"] = pd.to_datetime(df_25["endDate"])
df_25 = df_25.sort_values("endDate").reset_index(drop=True)
df_25[["market_id","endDate"]].head()

,market_id,endDate
0,249778,2023-04-02 00:00:00+00:00
1,250474,2023-04-26 00:00:00+00:00
2,248594,2023-05-01 00:00:00+00:00
3,251027,2023-05-22 00:00:00+00:00
4,251046,2023-05-28 00:00:00+00:00


In [14]:
df_25[["market_id","endDate"]].tail()

,market_id,endDate
38,254573,2024-12-30 12:00:00+00:00
39,254572,2024-12-30 12:00:00+00:00
40,254579,2024-12-30 12:00:00+00:00
41,254578,2024-12-30 12:00:00+00:00
42,254571,2024-12-31 00:00:00+00:00


In [15]:
split_idx = int(len(df_25) * 0.70)

train_df = df_25.iloc[:split_idx].copy()
test_df = df_25.iloc[split_idx:].copy()

print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (30, 34)
Test : (13, 34)


In [17]:
print(train_df["endDate"].min(),train_df["endDate"].max())

print(test_df["endDate"].min(),test_df["endDate"].max()
)

2023-04-02 00:00:00+00:00 2024-06-30 00:00:00+00:00
2024-06-30 00:00:00+00:00 2024-12-31 00:00:00+00:00


#  can we detect the regime on future mearkets ? 

In [18]:
features = [
    "early_last_probability",
    "early_probability_range",
    "early_realized_volatility",
    "early_trend",
    "early_distance_to_boundary",
    "early_max_drawdown",
    "early_reversals",
    "early_entropy"
]

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train = train_df[features]
y_train = train_df["target"]

X_test = test_df[features]
y_test = test_df["target"]

model = Pipeline([("scaler", StandardScaler()),("logreg", LogisticRegression())])

model.fit(X_train, y_train)
test_pred = model.predict(X_test)

print(classification_report(y_test, test_pred))

              precision    recall  f1-score   support

           0       0.20      1.00      0.33         1
           1       1.00      0.67      0.80        12

    accuracy                           0.69        13
   macro avg       0.60      0.83      0.57        13
weighted avg       0.94      0.69      0.76        13



In [19]:
from sklearn.metrics import accuracy_score

acc_test = accuracy_score(y_test,test_pred)

print(acc_test)

0.6923076923076923


In [20]:
train_df["target"].value_counts()

target
1    20
0    10
Name: count, dtype: int64

In [22]:
test_df["target"].value_counts()

target
1    12
0     1
Name: count, dtype: int64

In [23]:
(df_25.groupby(df_25["endDate"].dt.year)["market_regime"].value_counts())

endDate  market_regime         
2023     Anchored / Noisy           9
         Information Processing     7
2024     Information Processing    25
         Anchored / Noisy           2
Name: count, dtype: int64

In [24]:
pd.crosstab(df_25["endDate"].dt.year,df_25["market_regime"])

market_regime,Anchored / Noisy,Information Processing
endDate,,
2023,9,7
2024,2,25


In [25]:
regime_by_year = pd.crosstab(df_25["endDate"].dt.year,df_25["market_regime"],normalize="index") * 100
regime_by_year

market_regime,Anchored / Noisy,Information Processing
endDate,,
2023,56.250000,43.750000
2024,7.407407,92.592593


In [27]:
import plotly.express as px

fig = px.bar(regime_by_year.reset_index(),x="endDate",y=["Anchored / Noisy","Information Processing"],
    barmode="stack",
    title="Market Regime Composition Through Time"
)
fig.show()